In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install pyyaml

In [0]:
from databricks.sdk import WorkspaceClient
from utilities import load_input_csv
from sqlserver.connector import SQLServerConnector

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'



In [0]:
from core.runner import run_pipeline_generation, list_connectors, get_connector_info

# Connector to use (see supported connectors above)
connector_name = "sqlserver"

input_csv = 'examples/tapworks/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakehouse-tapworks/sqlserver/examples/tapworks/deployment'

targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

max_tables_per_pipeline = 100
max_tables_per_gateway = 100

# Run pipeline generation
result_df = run_pipeline_generation(
    connector_name=connector_name,
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
    override_config=override_config,
    max_tables_per_pipeline=max_tables_per_pipeline,
    max_tables_per_gateway=max_tables_per_gateway,
)

print(f"\nProcessed {len(result_df)} tables")
print(f"Pipeline groups: {result_df['pipeline_group'].nunique()}")
if 'gateway' in result_df.columns:
    print(f"Gateways: {result_df['gateway'].nunique()}")

## Example: Using Custom Default Values and Overrides

You can use `default_values` and `override_input_config` parameters for advanced configuration:

**`default_values` Parameter:**
- Provides custom default values for optional columns
- Merged with built-in defaults (your values take precedence)
- Applied only to missing/empty values in the CSV

**`override_input_config` Parameter:**
- Forces specific column values for ALL rows
- Overwrites any existing values in the CSV
- Useful for environment-specific overrides

**Example Use Cases:**
- Override schedule for testing (e.g., disable auto-runs)
- Force all pipelines to use specific gateway settings
- Set custom defaults for columns not in your CSV

## no worker/driver type specfied here to replace the empty ones in config -> gateway will use the platform defaults

## worker/driver type are specfied here to replace the empty ones in config -> gateway will use these specified ones